In [ ]:
import os
from pathlib import Path

import boto3
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.training.configs import Compute, InputData, SourceCode, StoppingCondition
from sagemaker.train.model_trainer import Mode, ModelTrainer

NOTEBOOK_PATH = Path(globals().get('__vsc_ipynb_file__', Path.cwd() / 'train.ipynb')).resolve()
TRAINING_DIR = NOTEBOOK_PATH.parent

from src.train.static_embedding.shared import (
    build_job_name,
    derive_job_basename,
    find_project_root,
    set_global_seed,
    prepare_dataset_files,
)

os.environ['SAGEMAKER_EXECUTION_ROLE_ARN'] = "arn:aws:iam::417153931970:role/mlops"

PROJECT_ROOT = find_project_root(TRAINING_DIR)

from src.preprocess.standardize.two_car_pros import DATASET_LANCEDB_DIR, LANCEDB_STANDARDIZED_RECORDS

SEED = int(os.getenv('SEED', '42'))
set_global_seed(SEED)

region = boto3.Session().region_name or os.getenv('AWS_DEFAULT_REGION', 'eu-west-1')
role = os.getenv('SAGEMAKER_EXECUTION_ROLE_ARN')
if not role:
    raise ValueError('SAGEMAKER_EXECUTION_ROLE_ARN must be set for AWS training mode.')

sess = Session()

JOB_BASENAME = derive_job_basename(TRAINING_DIR, project_root=PROJECT_ROOT)
JOB_NAME = build_job_name(TRAINING_DIR, project_root=PROJECT_ROOT)
WORK_DIR = TRAINING_DIR / 'artifacts' / JOB_BASENAME
DATA_DIR = WORK_DIR / 'input'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Training dir: {TRAINING_DIR}')
print(f'Job basename: {JOB_BASENAME}')
print(f'Example job name: {JOB_NAME}')
print(f'Region: {region}')

In [ ]:
train_pairs, eval_pairs, train_file, eval_file = prepare_dataset_files(
    db_dir=DATASET_LANCEDB_DIR,
    table_name=LANCEDB_STANDARDIZED_RECORDS,
    output_dir=DATA_DIR,
    seed=SEED,
    eval_ratio=0.2,
    max_pairs=10000,
)

train_channel_dir = train_file.parent
eval_channel_dir = eval_file.parent

print(f'Train file: {train_file}')
print(f'Eval file: {eval_file}')
print(f'Train rows: {len(train_pairs):,} | Eval rows: {len(eval_pairs):,}')

In [ ]:
from src.train.static_embedding.sagemaker.shared import HyperparametersModel
from src.train.static_embedding.shared import build_mlflow_environment, resolve_managed_mlflow_tracking_uri


hf_training_image = image_uris.retrieve(
    framework='huggingface',
    region=region,
    version='4.56.2',
    base_framework_version='pytorch2.8.0',
    py_version='py312',
    image_scope='training',
)

source_code = SourceCode(
    source_dir=str(TRAINING_DIR / 'sagemaker'),
    entry_script='launch.sh',
)

train_input = InputData(
    channel_name='train',
    data_source=str(train_channel_dir),
    content_type='application/jsonlines',
)
eval_input = InputData(
    channel_name='eval',
    data_source=str(eval_channel_dir),
    content_type='application/jsonlines',
)

hyperparameters = HyperparametersModel(
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-1,
    warmup_ratio=0.1,
    seed=SEED,
).model_dump_json()

mlflow_tracking_uri = resolve_managed_mlflow_tracking_uri(PROJECT_ROOT)
environment = build_mlflow_environment(
    mode='aws',
    experiment_name=JOB_BASENAME,
    tracking_uri=mlflow_tracking_uri,
)
print(f"MLflow tracking target: {environment['MLFLOW_TRACKING_URI']}")

instance_type = 'ml.g4dn.xlarge'
instance_count = 1
max_runtime_in_seconds = 14400
max_wait_time_in_seconds = max_runtime_in_seconds + 1800

print(f'Managed spot training enabled for {instance_type} x {instance_count}')
print(f'Max runtime: {max_runtime_in_seconds}s | Max wait: {max_wait_time_in_seconds}s')

trainer = ModelTrainer(
    training_mode=Mode.SAGEMAKER_TRAINING_JOB,
    training_image=hf_training_image,
    role=role,
    sagemaker_session=sess,
    source_code=source_code,
    compute=Compute(
        instance_count=instance_count,
        instance_type=instance_type,
        enable_managed_spot_training=True,
    ),
    stopping_condition=StoppingCondition(
        max_runtime_in_seconds=max_runtime_in_seconds,
        max_wait_time_in_seconds=max_wait_time_in_seconds,
    ),
    hyperparameters={'json': hyperparameters},
    base_job_name=JOB_BASENAME,
    environment=environment,
)

trainer.train(
    input_data_config=[train_input, eval_input],
    wait=True,
    logs=False,
)